# Experiment 2: Relevance-Aware DRAM Caching

**Baseline**: `M3-Baseline.ipynb` running `full_EdgeRAG.yaml` on `arch/basic8.yaml`  
(gte-base-en-v1.5 encoder + FAISS retrieval + Sheared-LLaMA-2.7B decoder, INT8, EdgeRAG platform)

## Hypotheses
1. Memory accesses dominate energy and latency at large retrieval scales.
2. DRAM caching reduces latency at the cost of higher energy.
3. There exists a break-even frontier where caching transitions from costing more energy
   than the no-cache baseline to costing less, as DRAM capacity and reuse ratio grow.

## What this notebook does
- Loads the AccelForge baseline from `baseline_cache.pkl` (produced by the last cell of `M3-Baseline.ipynb`).
- Reads per-bit access costs directly from `arch/basic8.yaml` (hard-coded constants; no full mapping run needed).
- Generates a synthetic Zipf-distributed document-access trace per BEIR reuse ratio (Table 2).
- Runs the triple sweep `(DRAM_GB, reuse_ratio, policy)` through `cache_sim.py`.
- Converts hit/miss counts to energy and latency and adds the non-SIM baseline.
- Produces heatmaps, the LFU-vs-OPT gap plot, and a hit-rate bar chart.

**Results are reported as-is. No interpretation of whether they support or contradict the hypotheses is made here.**

---
### Note on parameter scale
The plan specifies `N_DOCS = 10^5` (100 000) and `n_queries = 5 000`.  
At those settings, the corpus (`100 000 × 768 B = 76.8 MB`) fits entirely inside even the
smallest DRAM sweep point (1 GB → 1.4 M embedding slots), so the cache never fills and
the DRAM-size axis is flat.  
To see evictions and a non-trivial LFU-vs-OPT gap across the 1–16 GB sweep, this notebook
uses `N_DOCS_TRACE = 2 000 000` and `N_QUERIES = 100 000` for the *cache simulation*
while keeping `N_DOCS = 1 024` for the AccelForge baseline (as in M3-Baseline).  
You can override `N_DOCS_TRACE` and `N_QUERIES` in Cell 2 and re-run.

## Cell 1 — Imports

In [ ]:
import sys, os, pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.insert(0, '/home/workspace/workspace')
sys.path.insert(0, '/home/workspace/workspace/scripts')

from cache_sim import (
    synth_trace,
    simulate_no_cache,
    simulate_lfu_cost_aware,
    simulate_belady_opt,
    CacheResult,
)

%matplotlib inline
print('Imports OK')

## Cell 2 — Parameters

In [ ]:
# ── AccelForge workload parameters (must match M3-Baseline) ──────────────────
N_DOCS    = 1_024   # corpus size used in AccelForge (M3-Baseline default)
N_TOKENS  = 512     # query token length
K         = 10      # top-k docs retrieved per query
L         = 512     # tokens per doc

# ── Cache-simulation parameters ──────────────────────────────────────────────
# N_DOCS_TRACE is the corpus size used in the trace simulation.
# Set larger than N_DOCS so the DRAM sweep shows meaningful eviction dynamics.
N_DOCS_TRACE = 2_000_000   # 2 M docs × 768 B = 1.53 GB corpus
N_QUERIES    = 100_000     # number of queries in the simulated trace
K_SIM        = 20          # docs retrieved per query in the simulation
DECAY_FACTOR = 0.99        # EdgeRAG Algorithm 2 counter decay
SEED         = 42

# ── DRAM sweep ───────────────────────────────────────────────────────────────
DRAM_GB_VALUES = [1, 2, 4, 8, 16]   # GB

# ── Reuse ratios from BEIR Table 2 ───────────────────────────────────────────
# (dataset name, reuse_ratio)
BEIR_DATASETS = [
    ('nq',       1.25),
    ('hotpotqa', 1.42),
    ('scidocs',  1.73),
    ('quora',    1.91),
    ('fever',    2.41),
    ('fiqa',     4.47),
]
DATASET_NAMES  = [d for d, _ in BEIR_DATASETS]
REUSE_RATIOS   = [r for _, r in BEIR_DATASETS]

# ── Hardware constants from arch/basic8.yaml ──────────────────────────────────
# (These do not change with DRAM_SIZE_GB; only capacity changes.)
ENC_EMBED_DIM       = 768          # enc_embed_dim from full_EdgeRAG.yaml
BITS_PER_VAL        = 8            # INT8
EMB_BITS            = ENC_EMBED_DIM * BITS_PER_VAL  # bits per embedding = 6144

DRAM_ENERGY_pJ_per_bit  = 7.03e-12 * 1e12   # 7.03e-12 J/bit → in pJ
DRAM_BW_GBps            = 68.0              # LPDDR5 bandwidth in GB/s
DISK_ENERGY_pJ_per_bit  = 0.0               # disk energy = 0 in basic8.yaml
DISK_BW_GBps            = 7.0               # NVMe SSD bandwidth in GB/s

# Per-embedding access costs
#   HIT  → DRAM read
#   MISS → disk read (stream to DRAM) + DRAM write (cache it) + DRAM read (compute)
E_HIT_pJ   = DRAM_ENERGY_pJ_per_bit * EMB_BITS             # 1 DRAM read
E_MISS_pJ  = (DISK_ENERGY_pJ_per_bit * EMB_BITS            # disk read   (= 0)
              + DRAM_ENERGY_pJ_per_bit * EMB_BITS           # DRAM write (cache)
              + DRAM_ENERGY_pJ_per_bit * EMB_BITS)          # DRAM read  (compute)

L_HIT_ns   = EMB_BITS / (DRAM_BW_GBps * 1e9 * 8) * 1e9    # DRAM read latency
L_MISS_ns  = (EMB_BITS / (DISK_BW_GBps * 1e9 * 8) * 1e9   # disk read latency
              + EMB_BITS / (DRAM_BW_GBps * 1e9 * 8) * 1e9  # DRAM write latency
              + EMB_BITS / (DRAM_BW_GBps * 1e9 * 8) * 1e9) # DRAM read latency

print(f'Embedding size: {EMB_BITS} bits = {ENC_EMBED_DIM} bytes')
print(f'E_HIT  = {E_HIT_pJ:.1f} pJ   L_HIT  = {L_HIT_ns:.1f} ns')
print(f'E_MISS = {E_MISS_pJ:.1f} pJ   L_MISS = {L_MISS_ns:.1f} ns')
print()
print(f'Trace: N_DOCS_TRACE={N_DOCS_TRACE:,}  N_QUERIES={N_QUERIES:,}  K_SIM={K_SIM}')
print(f'Corpus size for trace: {N_DOCS_TRACE * ENC_EMBED_DIM / 1e9:.2f} GB')
print()
for gb in DRAM_GB_VALUES:
    cap = int(gb * 1024**3 / ENC_EMBED_DIM)
    pct = min(100.0, 100.0 * cap / N_DOCS_TRACE)
    print(f'  DRAM {gb:2d} GB → {cap:>10,} embedding slots ({pct:.0f}% of trace corpus)')

## Cell 3 — Load AccelForge baseline

In [ ]:
BASELINE_PATH = '/home/workspace/workspace/baseline_cache.pkl'

with open(BASELINE_PATH, 'rb') as f:
    bl = pickle.load(f)

baseline_energy_J  = bl['baseline_energy_J']   # einsum_name -> J
baseline_latency_s = bl['baseline_latency_s']  # einsum_name -> s
SIM_EINSUM         = bl['sim_einsum']          # 'doc_scores'

# Non-SIM totals: everything except the similarity/retrieval einsum.
# We replace that einsum's memory traffic with the simulator output.
non_sim_einsums = [e for e in baseline_energy_J if e != SIM_EINSUM]

E_nonsim_pJ = sum(baseline_energy_J[e] for e in non_sim_einsums) * 1e12
L_nonsim_ns = sum(baseline_latency_s[e] for e in non_sim_einsums) * 1e9

print(f'Loaded baseline from {BASELINE_PATH}')
print(f'SIM einsum: "{SIM_EINSUM}"')
print(f'Non-SIM einsums ({len(non_sim_einsums)}): {non_sim_einsums[:5]} ...')
print(f'Non-SIM total energy: {E_nonsim_pJ:.0f} pJ')
print(f'Non-SIM total latency: {L_nonsim_ns:.0f} ns')

## Cell 4 — Synthesize access traces (one per BEIR reuse ratio)

In [ ]:
print('Generating Zipf traces...')
traces = {}
for name, reuse in BEIR_DATASETS:
    t = synth_trace(
        n_queries=N_QUERIES,
        n_docs=N_DOCS_TRACE,
        reuse_ratio=reuse,
        k_per_query=K_SIM,
        seed=SEED,
    )
    unique = len(set(t))
    actual_reuse = len(t) / unique if unique > 0 else float('inf')
    traces[name] = t
    print(f'  {name:<10} reuse target={reuse:.2f}  '
          f'actual={actual_reuse:.2f}  '
          f'unique={unique:,}  total={len(t):,}')

print('Done.')

## Cell 5 — Run the triple sweep (DRAM_GB × reuse_ratio × policy)

In [ ]:
POLICIES = ['no_cache', 'lfu', 'opt']

# results[dram_gb][dataset_name][policy] = CacheResult
sim_results = {gb: {name: {} for name, _ in BEIR_DATASETS} for gb in DRAM_GB_VALUES}

# Uniform gen_latency (same for every document — see cache_sim.py docstring)
gen_latency_uniform = {}  # empty dict → cache_sim defaults to 1.0 per doc

total = len(DRAM_GB_VALUES) * len(BEIR_DATASETS)
done  = 0

for gb in DRAM_GB_VALUES:
    capacity = int(gb * 1024**3 / ENC_EMBED_DIM)  # embedding slots
    for name, reuse in BEIR_DATASETS:
        trace = traces[name]
        sim_results[gb][name]['no_cache'] = simulate_no_cache(trace)
        sim_results[gb][name]['lfu']      = simulate_lfu_cost_aware(
            trace, capacity, gen_latency_uniform, decay_factor=DECAY_FACTOR
        )
        sim_results[gb][name]['opt']      = simulate_belady_opt(trace, capacity)
        done += 1
        print(f'  [{done}/{total}] DRAM={gb} GB  dataset={name}')

print('Sweep complete.')

## Cell 6 — Convert hit/miss counts to energy, latency, and EDP per query

In [ ]:
# For each (DRAM_GB, dataset, policy) triple, compute per-query totals:
#   energy (pJ) = non-SIM baseline + hits*E_HIT + misses*E_MISS
#   latency (ns) = non-SIM baseline + hits*L_HIT + misses*L_MISS
#   EDP = energy * latency

def per_query_metrics(result: CacheResult, n_queries: int):
    h = result.hits   / n_queries
    m = result.misses / n_queries
    e_pJ  = E_nonsim_pJ + h * E_HIT_pJ  + m * E_MISS_pJ
    l_ns  = L_nonsim_ns + h * L_HIT_ns  + m * L_MISS_ns
    edp   = e_pJ * l_ns
    return e_pJ, l_ns, edp

# Build 2-D arrays indexed [dram_gb_idx, dataset_idx] for each policy.
ndram = len(DRAM_GB_VALUES)
ndata = len(BEIR_DATASETS)

energy  = {p: np.zeros((ndram, ndata)) for p in POLICIES}
latency = {p: np.zeros((ndram, ndata)) for p in POLICIES}
edp     = {p: np.zeros((ndram, ndata)) for p in POLICIES}
hitrate = {p: np.zeros((ndram, ndata)) for p in POLICIES}

for i, gb in enumerate(DRAM_GB_VALUES):
    for j, (name, _) in enumerate(BEIR_DATASETS):
        for p in POLICIES:
            res = sim_results[gb][name][p]
            e, l, ed = per_query_metrics(res, N_QUERIES)
            energy[p][i, j]  = e
            latency[p][i, j] = l
            edp[p][i, j]     = ed
            hitrate[p][i, j] = res.hit_rate

print('Energy / latency / EDP matrices built.')
print()
print('Sample: no_cache energy [pJ] (should be same across DRAM sizes):')
print(np.array_str(energy['no_cache'], precision=0, suppress_small=True))

## Cell 7 — Heatmaps: energy and latency per policy

In [ ]:
POLICY_LABELS = {'no_cache': 'No Cache', 'lfu': 'LFU (EdgeRAG)', 'opt': 'Bélády OPT'}
DRAM_LABELS   = [f'{g} GB' for g in DRAM_GB_VALUES]

def draw_heatmap(ax, data, title, cmap, fmt, vmin=None, vmax=None):
    im = ax.imshow(data, cmap=cmap, aspect='auto',
                   vmin=vmin if vmin is not None else data.min(),
                   vmax=vmax if vmax is not None else data.max())
    ax.set_xticks(range(ndata))
    ax.set_xticklabels(DATASET_NAMES, rotation=30, ha='right', fontsize=8)
    ax.set_yticks(range(ndram))
    ax.set_yticklabels(DRAM_LABELS, fontsize=8)
    ax.set_title(title, fontsize=9)
    # Annotate cells
    for i in range(ndram):
        for j in range(ndata):
            ax.text(j, i, fmt.format(data[i, j]),
                    ha='center', va='center', fontsize=6.5,
                    color='white' if data[i, j] > (data.max() * 0.6) else 'black')
    return im

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
fig.suptitle('Experiment 2: Energy and Latency per Query\n'
             f'(DRAM sweep × BEIR reuse ratio, N_DOCS_TRACE={N_DOCS_TRACE:,}, N_QUERIES={N_QUERIES:,})',
             fontsize=11)

# Shared colour scales within each row (energy / latency)
e_max = max(energy[p].max() for p in POLICIES)
l_max = max(latency[p].max() for p in POLICIES)

for col, policy in enumerate(POLICIES):
    ax_e = axes[0, col]
    ax_l = axes[1, col]

    im_e = draw_heatmap(ax_e, energy[policy] / 1e3,   # pJ → nJ
                        f'Energy — {POLICY_LABELS[policy]}\n[nJ/query]',
                        'YlOrRd', '{:.0f}',
                        vmin=0, vmax=e_max / 1e3)
    im_l = draw_heatmap(ax_l, latency[policy] / 1e3,  # ns → µs
                        f'Latency — {POLICY_LABELS[policy]}\n[µs/query]',
                        'YlGnBu', '{:.1f}',
                        vmin=0, vmax=l_max / 1e3)

    if col == 0:
        ax_e.set_ylabel('DRAM size', fontsize=9)
        ax_l.set_ylabel('DRAM size', fontsize=9)

fig.colorbar(im_e, ax=axes[0, :], shrink=0.6, label='nJ/query')
fig.colorbar(im_l, ax=axes[1, :], shrink=0.6, label='µs/query')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('/home/workspace/workspace/exp2_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved exp2_heatmaps.png')

## Cell 8 — LFU-vs-OPT headroom gap plot

In [ ]:
# gap[i,j] = (E_LFU - E_OPT) / (E_nocache - E_OPT)
# Fraction of the no-cache → OPT energy range captured by LFU.
# 0 = LFU achieves OPT; 1 = LFU is no better than no cache.
# Clamp to [0, 1] to handle numerical noise.

denominator = energy['no_cache'] - energy['opt']
# Avoid division by zero (can occur when no evictions happen and all
# policies give the same result).
safe_denom  = np.where(np.abs(denominator) < 1e-6, np.nan, denominator)
gap         = np.clip((energy['lfu'] - energy['opt']) / safe_denom, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Experiment 2: LFU-vs-OPT Energy Gap  &  Hit Rates', fontsize=11)

# --- Left: gap line plot ---
ax = axes[0]
colors = plt.cm.plasma(np.linspace(0.15, 0.85, ndram))
for i, (gb, c) in enumerate(zip(DRAM_GB_VALUES, colors)):
    row = gap[i, :]
    ax.plot(DATASET_NAMES, row, marker='o', color=c, label=f'{gb} GB DRAM')
ax.set_xlabel('BEIR dataset (increasing reuse ratio →)', fontsize=9)
ax.set_ylabel('(E_LFU − E_OPT) / (E_nocache − E_OPT)', fontsize=9)
ax.set_title('LFU headroom gap (0 = LFU matches OPT, 1 = LFU = no cache)', fontsize=9)
ax.set_ylim(-0.05, 1.05)
ax.axhline(0, ls='--', lw=0.8, color='gray')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=20)

# --- Right: hit-rate bar chart at representative operating point ---
# Representative: first DRAM size (1 GB) across all reuse ratios, all policies.
REP_DRAM_IDX = 0  # 1 GB
rep_gb = DRAM_GB_VALUES[REP_DRAM_IDX]

ax2 = axes[1]
x   = np.arange(ndata)
w   = 0.25
policy_colors = {'no_cache': '#e07070', 'lfu': '#70aae0', 'opt': '#70c87a'}
for k, policy in enumerate(POLICIES):
    ax2.bar(x + k * w, hitrate[policy][REP_DRAM_IDX, :],
            width=w, label=POLICY_LABELS[policy],
            color=policy_colors[policy], edgecolor='white', linewidth=0.5)
ax2.set_xticks(x + w)
ax2.set_xticklabels(DATASET_NAMES, rotation=20, ha='right', fontsize=8)
ax2.set_ylabel('Cache hit rate', fontsize=9)
ax2.set_title(f'Hit rate at DRAM = {rep_gb} GB', fontsize=9)
ax2.set_ylim(0, 1.05)
ax2.legend(fontsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('/home/workspace/workspace/exp2_gap_hitrate.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved exp2_gap_hitrate.png')

## Cell 9 — Raw numbers table

In [ ]:
import pandas as pd

rows = []
for i, gb in enumerate(DRAM_GB_VALUES):
    for j, (name, reuse) in enumerate(BEIR_DATASETS):
        for p in POLICIES:
            res = sim_results[gb][name][p]
            rows.append({
                'DRAM_GB':    gb,
                'dataset':    name,
                'reuse':      reuse,
                'policy':     p,
                'hits':       res.hits,
                'misses':     res.misses,
                'hit_rate':   f'{res.hit_rate:.3f}',
                'energy_nJ':  f'{energy[p][i,j]/1e3:.1f}',
                'latency_us': f'{latency[p][i,j]/1e3:.2f}',
                'EDP':        f'{edp[p][i,j]:.2e}',
            })

df = pd.DataFrame(rows)
pd.set_option('display.max_rows', 120)
print(df.to_string(index=False))